<a target="_blank" rel="noopener noreferrer" href="https://colab.research.google.com/github/ccaudek/ds4psy_2023/blob/main/425_hierarchical_models.ipynb">![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)</a>

(stan-hier_beta_binom_model)=
# Modello gerarchico beta-binomiale con Stan

In questo capitolo, ripeteremo l'esercizio discusso nel capitolo {ref}`hier_beta_binom_model` relativo al concetto di modello gerarchico bayesiano.

In [1]:
import requests
from io import StringIO
from scipy.stats import gaussian_kde
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import arviz as az
import warnings
from cmdstanpy import cmdstan_path, CmdStanModel

/opt/anaconda3/envs/stan_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")

## Analisi bayesiana della "terapia tattile" 

Lo studio condotto da [Rosa et al. (1998)](https://doi.org/10.2307/1382936) esamina la pratica infermieristica del "Terapeutic Touch" (TT), la quale si basa sulla manipolazione di un presunto campo energetico del paziente. Gli autori indagano sull'affermazione fondamentale dei praticanti di TT riguardo alla loro capacità di percepire i campi energetici senza vedere.

Per testare questa affermazione, è stato condotto un esperimento in cui i praticanti dovevano identificare quale mano l'esaminatore aveva scelto senza guardare, attraverso l'uso di un pannello divisorio. L'esperimento coinvolgeva 21 operatori, ciascuno sottoposto a 10 prove, con 7 di essi sottoposti a un retest dopo un anno, totalizzando così 28 osservazioni complessive. L'obiettivo era determinare se la proporzione di risposte corrette superasse il 50%, che rappresenta la probabilità di successo casuale, e indagare le variazioni individuali nelle prestazioni.

Importiamo i dati forniti da {cite:t}`doing_bayesian_data_an`.

In [3]:
# Define the URL of the CSV file on GitHub
url = "https://raw.githubusercontent.com/boboppie/kruschke-doing_bayesian_data_analysis/master/2e/TherapeuticTouchData.csv"
# Download the content of the CSV file
response = requests.get(url)
tt_dat = pd.read_csv(StringIO(response.text))
print(tt_dat.head())

   y    s
0  1  S01
1  0  S01
2  0  S01
3  0  S01
4  0  S01


In [4]:
tt_dat.shape

(280, 2)

Nella colonna `y`, il valore 1 indica una risposta corretta, mentre 0 indica una risposta errata. La seconda colonna contiene il codice identificativo di ciascun operatore.

Calcoliamo la proporzione di risposte corrette per ciascun operatore.

In [4]:
tt_agg = tt_dat.groupby("s").agg(proportion_correct=("y", "mean")).reset_index()
tt_agg

,s,proportion_correct
0,S01,0.1
1,S02,0.2
2,S03,0.3
3,S04,0.3
4,S05,0.3
5,S06,0.3
6,S07,0.3
7,S08,0.3
8,S09,0.3
9,S10,0.3


Costruiamo il modello gerarchico partendo dall'assunzione che il numero di risposte corrette per ciascun operatore sia una variabile casuale Binomiale. Per ciascuno dei ventotto operatori possiamo dunque scrivere:

$$
y_i \sim Binomial(n_i, p_i),
$$

con $i = 0, \dots, 27$.

Quale distribuzione a priori per il parametro sconosciuto $p_i$ possiamo fissare una distribuzione Beta di parametri $a$ e $b$:

$$
p_i \sim Beta(a, b).
$$

Va notato che, in questo modo, gli iperparametri $a$ e $b$ sono condivisi tra tutti gli operatori. Se $a$ e $b$ sono noti, allora la distribuzione a posteriori per il parametro $p$ è una distribuzione Beta:

$$
p_i \mid y_i \sim Beta(a + y_i, b + n_i - y_i). 
$$

Nel caso generale in cui gli iperparametri $a$ e $b$ sono incogniti, è necessario fissare una distribuzione a priori per ciascuno di essi. 

Applichiamo dunque il modello gerarchico descritto sopra ai dati del *therapeutic touch*. Iniziamo a calcolare il numero di risposte corrette di ciascun operatore.

In [5]:
result = tt_dat.groupby("s")["y"].sum().reset_index()
y = result["y"]
print(*y)

1 2 3 3 3 3 3 3 3 3 4 4 4 4 4 5 5 5 5 5 5 5 6 6 7 7 7 8


Creiamo inoltre il vettore `N` che fornisce il numero di prove per ciascun operatore.

In [6]:
N = tt_dat.groupby("s")["y"].count() 

Esaminiamo dunque i dati a disposizione.

In [7]:
print(*N)

10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10


In [9]:
print(*y)

1 2 3 3 3 3 3 3 3 3 4 4 4 4 4 5 5 5 5 5 5 5 6 6 7 7 7 8


In [10]:
type(y)

pandas.core.series.Series

Creaiamo un dizionario con i dati necessari per Stan.

In [8]:
data = {
    "N" : 28,
    "y" : y.tolist(),
    "n_trials" : N.tolist()
}

print(data)

{'N': 28, 'y': [1, 2, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 6, 6, 7, 7, 7, 8], 'n_trials': [10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]}


Leggiamo il codice Stan che implementa il modello descritto nel capitolo {ref}`hier_beta_binom_model`.

In [9]:
stan_file = os.path.join('stan', 'h_beta_binom_model.stan')
with open(stan_file, 'r') as f:
    print(f.read())

data {
  int<lower=0> N;  // Number of participants
  array[N] int<lower=0> y;  // Number of successes for each participant
  array[N] int<lower=0> n_trials;  // Number of trials for each participant
}

parameters {
  real<lower=0> alpha;  // Alpha parameter for the Beta distribution
  real<lower=0> beta;   // Beta parameter for the Beta distribution
  array[N] real<lower=0, upper=1> p;  // Success probability for each participant
}

model {
  // Priors
  alpha ~ gamma(8, 2);
  beta ~ gamma(27, 5);
  
  // Each participant's success probability follows a Beta distribution
  p ~ beta(alpha, beta);
  
  // Likelihood of the observed data
  for (i in 1:N) {
    y[i] ~ binomial(n_trials[i], p[i]);
  }
}



### Sezione `data`
- **`N`**: rappresenta il numero totale di partecipanti allo studio.
- **`y`**: un array che contiene il numero di successi osservati per ciascun partecipante.
- **`n_trials`**: un array che indica il numero di tentativi compiuti da ogni partecipante.

Queste sono le osservazioni raccolte dall'esperimento e rappresentano la base dati su cui verrà adattato il modello.

### Sezione `parameters`
- **`alpha` e `beta`**: sono i parametri della distribuzione Beta, che servono a modellare la variabilità della probabilità di successo tra i partecipanti. Questi parametri determinano la forma della distribuzione Beta, influenzando direttamente la variabilità e la media delle probabilità di successo `p`.
- **`p`**: un array che rappresenta la probabilità di successo individuale per ciascun partecipante. Queste probabilità sono stimate dal modello e sono soggette a variabilità tra i partecipanti.

### Sezione `model`
- **Priori**: 
  - `alpha ~ gamma(8, 2)`: indica che il parametro `alpha` segue una distribuzione Gamma con forma 8 e tasso 2. Questo è un prior informativo che suggerisce una certa credenza iniziale sulla distribuzione delle probabilità di successo.
  - `beta ~ gamma(27, 5)`: simile ad `alpha`, anche `beta` segue una distribuzione Gamma, ma con parametri diversi, riflettendo una diversa credenza iniziale sulla distribuzione delle probabilità di fallimento.
  
  Questi priori definiscono una distribuzione Beta flessibile che può adattarsi a vari tipi di dati di conteggio successo-fallimento.

- **Distribuzione delle probabilità di successo**:
  - `p ~ beta(alpha, beta)`: indica che la probabilità di successo `p` per ogni partecipante segue una distribuzione Beta parametrizzata da `alpha` e `beta`. Questo legame gerarchico permette di condividere informazioni tra i partecipanti, stabilendo che, sebbene ogni partecipante abbia la sua probabilità di successo, queste probabilità provengono dalla stessa distribuzione Beta.
  
- **Likelihood**:
  - Il ciclo `for (i in 1:N)` itera su ciascun partecipante, dove `y[i] ~ binomial(n_trials[i], p[i])` modella il numero di successi `y[i]` per il partecipante `i` come una variabile casuale binomiale, con `n_trials[i]` tentativi e probabilità di successo `p[i]`. Questo collega direttamente la distribuzione delle probabilità di successo individuali alla distribuzione osservata dei successi.

Il modello gerarchico beta-binomiale adotta un approccio bayesiano per caratterizzare la variazione individuale nelle probabilità di successo all'interno di un contesto di prove binomiali. Questo modello incorpora una struttura gerarchica, dove i parametri alpha e beta regolano la distribuzione delle probabilità p tramite una distribuzione Beta. Tale struttura consente di catturare sia le differenze individuali sia le informazioni comuni tra i partecipanti. Di conseguenza, il modello risulta particolarmente idoneo per analizzare dati caratterizzati da variazioni intra-gruppo in contesti sperimentali o osservazionali basati su frequenze.

Compiliamo il modello.

In [10]:
model = CmdStanModel(stan_file=stan_file)
print(model)

20:22:36 - cmdstanpy - INFO - compiling stan file /Users/corradocaudek/_repositories/ds4p/chapter_4/stan/h_beta_binom_model.stan to exe file /Users/corradocaudek/_repositories/ds4p/chapter_4/stan/h_beta_binom_model
20:22:45 - cmdstanpy - INFO - compiled model executable: /Users/corradocaudek/_repositories/ds4p/chapter_4/stan/h_beta_binom_model


CmdStanModel: name=h_beta_binom_model
	 stan_file=/Users/corradocaudek/_repositories/ds4p/chapter_4/stan/h_beta_binom_model.stan
	 exe_file=/Users/corradocaudek/_repositories/ds4p/chapter_4/stan/h_beta_binom_model
	 compiler_options=stanc_options={}, cpp_options={}


Eseguiamo il campionamento MCMC.

In [11]:
fit = model.sample(
    data=data,
    iter_sampling = 4000,
    iter_warmup = 2000,
    seed = 84735,
    chains = 4
)

20:22:50 - cmdstanpy - INFO - CmdStan start processing
chain 1 |          | 00:00 Status


chain 1 |██▌       | 00:00 Iteration: 1400 / 6000 [ 23%]  (Warmup)


chain 1 |█████▋    | 00:00 Iteration: 3200 / 6000 [ 53%]  (Sampling)


chain 1 |████████▌ | 00:00 Iteration: 5000 / 6000 [ 83%]  (Sampling)


chain 1 |██████████| 00:00 Sampling completed                       
chain 2 |██████████| 00:00 Sampling completed                       
chain 3 |██████████| 00:00 Sampling completed                       
chain 4 |██████████| 00:00 Sampling completed                       


20:22:50 - cmdstanpy - INFO - CmdStan done processing.
20:22:50 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: gamma_lpdf: Random variable is inf, but must be positive finite! (in 'h_beta_binom_model.stan', line 16, column 2 to column 22)
Exception: gamma_lpdf: Random variable is inf, but must be positive finite! (in 'h_beta_binom_model.stan', line 16, column 2 to column 22)
Consider re-running with show_console=True if the above output is unclear!


Esaminiamo le stime a posteriori dei parametri.

In [12]:
az.summary(fit, var_names=(["alpha", "beta", "p"]), hdi_prob=0.95)

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,4.576,0.869,2.963,6.296,0.008,0.006,11649.0,10746.0,1.0
beta,5.750,0.934,3.950,7.532,0.008,0.006,13545.0,11299.0,1.0
p[0],0.272,0.099,0.089,0.466,0.001,0.000,20382.0,11486.0,1.0
p[1],0.323,0.105,0.133,0.533,0.001,0.001,22305.0,11089.0,1.0
p[2],0.372,0.107,0.165,0.581,0.001,0.001,23383.0,11481.0,1.0
p[3],0.373,0.107,0.173,0.586,0.001,0.001,23381.0,11867.0,1.0
p[4],0.371,0.107,0.168,0.578,0.001,0.001,21497.0,11252.0,1.0
p[5],0.373,0.107,0.166,0.579,0.001,0.001,24071.0,11539.0,1.0
p[6],0.372,0.105,0.176,0.578,0.001,0.000,24243.0,12461.0,1.0
p[7],0.372,0.107,0.169,0.579,0.001,0.001,22363.0,11482.0,1.0


I risultati ottenuti replicano quelli trovati con PyMC.

## Watermark

In [13]:
%load_ext watermark
%watermark -n -u -v -iv -w -m -p cmdstanpy 

Last updated: Tue Mar 12 2024

Python implementation: CPython
Python version       : 3.12.2
IPython version      : 8.22.2

cmdstanpy: 1.2.1

Compiler    : Clang 16.0.6 
OS          : Darwin
Release     : 23.4.0
Machine     : arm64
Processor   : arm
CPU cores   : 8
Architecture: 64bit

arviz     : 0.17.0
requests  : 2.31.0
seaborn   : 0.13.2
matplotlib: 3.8.3
scipy     : 1.12.0
pandas    : 2.2.1
numpy     : 1.26.4

Watermark: 2.4.3

